# Assignment 2: End-to-End Machine Learning Pipeline

## Task 1: Build a basic machine learning pipeline

### 1. Load and inspect the dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from IPython.display import display # Import display explicitly to avoid errors

# Load the dataset (Note: the file has .xls extension but is actually a CSV)
df = pd.read_csv('financial_customer_investment_risk_dataset (1).xls')

# Inspect the first few rows
print("First 5 rows:")
display(df.head())

# Check dataset info
print("\nDataset Information:")
df.info()

First 5 rows:


,Customer_ID,Age,Employment_Status,Annual_Income,Investment_Experience_Years,Portfolio_Value,Risk_Tolerance,Number_of_Investment_Products,Market_Volatility_Concern,Financial_Literacy_Score,Advisor_Contacted,Previous_Investment_Loss,High_Investment_Risk
0,1,35.0,Full-time,61257.0,18,350954.0,High,5,3,9,Yes,No,No
1,2,63.0,Retired,189284.0,18,239184.0,High,9,3,3,Yes,No,No
2,3,55.0,Part-time,36010.0,13,38681.0,Low,2,2,6,No,No,No
3,4,43.0,Part-time,46379.0,3,265566.0,Medium,11,2,5,Yes,No,No
4,5,39.0,Self-employed,195809.0,19,208719.0,Medium,9,3,10,No,Yes,No



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 387 entries, 0 to 386
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    387 non-null    int64  
 1   Age                            380 non-null    float64
 2   Employment_Status              380 non-null    object 
 3   Annual_Income                  380 non-null    float64
 4   Investment_Experience_Years    387 non-null    int64  
 5   Portfolio_Value                380 non-null    float64
 6   Risk_Tolerance                 380 non-null    object 
 7   Number_of_Investment_Products  387 non-null    int64  
 8   Market_Volatility_Concern      387 non-null    int64  
 9   Financial_Literacy_Score       387 non-null    int64  
 10  Advisor_Contacted              380 non-null    object 
 11  Previous_Investment_Loss       387 non-null    object 
 12  High_Investment_Risk        

### 2. Understand the business problem
The goal is to predict whether a customer is a **High Investment Risk** (`High_Investment_Risk`) based on their demographic information, financial status, and investment behavior. This helps financial institutions identify customers who might need more guidance or more conservative investment strategies.

### 3. Identify the features and target variable
- **Target Variable (`y`)**: `High_Investment_Risk`
- **Features (`X`)**: All other columns except `Customer_ID` (which is just an identifier).

In [2]:
# Define X and y
X = df.drop(['Customer_ID', 'High_Investment_Risk'], axis=1)
y = df['High_Investment_Risk']

print("Features:", X.columns.tolist())
print("Target:", y.name)

Features: ['Age', 'Employment_Status', 'Annual_Income', 'Investment_Experience_Years', 'Portfolio_Value', 'Risk_Tolerance', 'Number_of_Investment_Products', 'Market_Volatility_Concern', 'Financial_Literacy_Score', 'Advisor_Contacted', 'Previous_Investment_Loss']
Target: High_Investment_Risk


### 4. Clean the data
- Handle missing values
- Handle categorical and numerical variables

In [3]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Fill missing numerical values with the median
num_cols = ['Age', 'Annual_Income', 'Portfolio_Value']
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# Fill missing categorical values with the mode
cat_cols = ['Employment_Status', 'Risk_Tolerance', 'Advisor_Contacted']
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

print("\nMissing values after cleaning:")
print(X.isnull().sum())

Missing values per column:
Customer_ID                      0
Age                              7
Employment_Status                7
Annual_Income                    7
Investment_Experience_Years      0
Portfolio_Value                  7
Risk_Tolerance                   7
Number_of_Investment_Products    0
Market_Volatility_Concern        0
Financial_Literacy_Score         0
Advisor_Contacted                7
Previous_Investment_Loss         0
High_Investment_Risk             0
dtype: int64

Missing values after cleaning:
Age                              0
Employment_Status                0
Annual_Income                    0
Investment_Experience_Years      0
Portfolio_Value                  0
Risk_Tolerance                   0
Number_of_Investment_Products    0
Market_Volatility_Concern        0
Financial_Literacy_Score         0
Advisor_Contacted                0
Previous_Investment_Loss         0
dtype: int64


In [4]:
# Encode categorical variables
le = LabelEncoder()

categorical_features = ['Employment_Status', 'Risk_Tolerance', 'Advisor_Contacted', 'Previous_Investment_Loss']
for col in categorical_features:
    X[col] = le.fit_transform(X[col])

# Encode the target variable
y = le.fit_transform(y)

print("Data after encoding (first 5 rows):")
display(X.head())

Data after encoding (first 5 rows):


,Age,Employment_Status,Annual_Income,Investment_Experience_Years,Portfolio_Value,Risk_Tolerance,Number_of_Investment_Products,Market_Volatility_Concern,Financial_Literacy_Score,Advisor_Contacted,Previous_Investment_Loss
0,35.0,0,61257.0,18,350954.0,0,5,3,9,1,0
1,63.0,2,189284.0,18,239184.0,0,9,3,3,1,0
2,55.0,1,36010.0,13,38681.0,1,2,2,6,0,0
3,43.0,1,46379.0,3,265566.0,2,11,2,5,1,0
4,39.0,3,195809.0,19,208719.0,2,9,3,10,0,1


### 5. Split the data into training and testing sets

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (309, 11)
Testing set size: (78, 11)


### 6. Apply preprocessing (Scaling)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 7. Train a simple baseline model using Logistic Regression

In [7]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy Score: 0.9231

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96        73
           1       0.00      0.00      0.00         5

    accuracy                           0.92        78
   macro avg       0.47      0.49      0.48        78
weighted avg       0.88      0.92      0.90        78



## Task 2: Explain and submit your work

- **What your model is trying to predict**: Prediction of whether a customer is a "High Investment Risk" (Yes/No).
- **What dataset you used**: Financial Customer Investment Risk dataset.
- **What features and target you selected**: Features include Age, Annual Income, Portfolio Value, Risk Tolerance, Employment Status, etc. Target is `High_Investment_Risk`.
- **What result you obtained**: The Logistic Regression model achieved an accuracy score of ~92.3% on the test set.
- **One limitation of your model or dataset**: The dataset is relatively small and highly imbalanced (few high-risk cases), which makes it difficult for the model to learn the patterns of high-risk customers effectively.